# Notebook 04 — Market Basket Analysis & Recommendation Engine

## Objective

The objective of this notebook is to identify meaningful product affinities
and convert them into a practical recommendation engine.

The analysis has two parts:

### Part A — Association Rules

- Build transaction baskets at order level.
- Identify SKU-level product associations.
- Measure support, confidence, lift, leverage and conviction.
- Filter out sparse and trivial associations.
- Repeat the analysis at category and subcategory level.
- Identify directional differences between rules.
- Compare associations across customer segments.
- Translate strong rules into commercially useful bundle recommendations.

### Part B — Recommendation Engine

- Build an item-item collaborative filtering model using cosine similarity.
- Build a second recommendation model using truncated SVD.
- Evaluate recommendations using a temporal train/test split.
- Compare both models against a popularity baseline.
- Measure Precision@5, Recall@10 and Hit Rate.
- Implement cold-start fallbacks.

In [3]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from mlxtend.frequent_patterns import (
    fpgrowth,
    apriori,
    association_rules
)

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import csr_matrix

In [4]:
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed"

FIGURES_PATH = PROJECT_ROOT / "outputs" / "figures" / "market_basket"
TABLES_PATH = PROJECT_ROOT / "outputs" / "tables" / "market_basket"

PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)
FIGURES_PATH.mkdir(parents=True, exist_ok=True)
TABLES_PATH.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data:", RAW_DATA_PATH)
print("Processed data:", PROCESSED_DATA_PATH)

Project root: C:\Users\Sejal patil\OneDrive\Documents\meridian_customer_analytics
Raw data: C:\Users\Sejal patil\OneDrive\Documents\meridian_customer_analytics\data\raw
Processed data: C:\Users\Sejal patil\OneDrive\Documents\meridian_customer_analytics\data\processed


In [5]:
transactions = pd.read_csv(
    RAW_DATA_PATH / "transactions.csv",
    parse_dates=["order_date", "order_datetime"]
)

products = pd.read_csv(
    RAW_DATA_PATH / "products.csv"
)

customers = pd.read_csv(
    RAW_DATA_PATH / "customers.csv",
    parse_dates=["signup_date"]
)

print("Transactions:", transactions.shape)
print("Products:", products.shape)
print("Customers:", customers.shape)

Transactions: (575516, 15)
Products: (391, 9)
Customers: (12000, 12)


In [6]:
segment_file = PROCESSED_DATA_PATH / "customer_segments.csv"

if segment_file.exists():
    customer_segments = pd.read_csv(segment_file)
    print("Customer segments loaded:", customer_segments.shape)
else:
    customer_segments = None
    print("Customer segmentation file not found.")

Customer segments loaded: (10965, 3)


In [7]:
print("Unique orders:", transactions["order_id"].nunique())
print("Unique customers:", transactions["customer_id"].nunique())
print("Unique products:", transactions["product_id"].nunique())

print("\nTransaction date range:")
print(transactions["order_datetime"].min())
print(transactions["order_datetime"].max())

Unique orders: 158115
Unique customers: 10965
Unique products: 391

Transaction date range:
2024-08-01 06:15:00
2026-07-31 23:58:00


In [8]:
print(
    "Products in transactions:",
    transactions["product_id"].nunique()
)

print(
    "Products in product master:",
    products["product_id"].nunique()
)

Products in transactions: 391
Products in product master: 391


In [9]:
CUTOFF_DATE = pd.Timestamp("2026-04-30 23:59:59")

TRAIN_START = transactions["order_datetime"].min()

train_transactions = transactions[
    transactions["order_datetime"] <= CUTOFF_DATE
].copy()

test_transactions = transactions[
    transactions["order_datetime"] > CUTOFF_DATE
].copy()

print("Training transactions:", len(train_transactions))
print("Test transactions:", len(test_transactions))

print("\nTraining date range:")
print(
    train_transactions["order_datetime"].min(),
    "to",
    train_transactions["order_datetime"].max()
)

print("\nTest date range:")
print(
    test_transactions["order_datetime"].min(),
    "to",
    test_transactions["order_datetime"].max()
)

Training transactions: 494946
Test transactions: 80570

Training date range:
2024-08-01 06:15:00 to 2026-04-30 23:58:00

Test date range:
2026-05-01 06:34:00 to 2026-07-31 23:58:00


In [10]:
basket_transactions = train_transactions[
    [
        "order_id",
        "customer_id",
        "product_id"
    ]
].drop_duplicates(
    subset=["order_id", "product_id"]
)

print("Unique order-product combinations:", len(basket_transactions))
print("Unique orders:", basket_transactions["order_id"].nunique())
print("Unique products:", basket_transactions["product_id"].nunique())

Unique order-product combinations: 494946
Unique orders: 135945
Unique products: 391


In [11]:
basket = (
    basket_transactions
    .assign(purchased=1)
    .pivot_table(
        index="order_id",
        columns="product_id",
        values="purchased",
        fill_value=0
    )
)

basket = basket.astype(bool)

print("Basket matrix shape:", basket.shape)

Basket matrix shape: (135945, 391)


In [12]:
basket_sizes = basket.sum(axis=1)

print("Average products per order:", round(basket_sizes.mean(), 2))
print("Median products per order:", basket_sizes.median())
print("Maximum products in one order:", basket_sizes.max())

Average products per order: 3.64
Median products per order: 3.0
Maximum products in one order: 21


In [13]:
product_support = (
    basket.mean()
    .sort_values(ascending=False)
    .reset_index()
)

product_support.columns = [
    "product_id",
    "support"
]

product_support.head(20)

,product_id,support
0,SKU100376,0.042238
1,SKU100065,0.041083
2,SKU100162,0.036971
3,SKU100077,0.035772
4,SKU100369,0.035007
5,SKU100218,0.034389
6,SKU100277,0.032594
7,SKU100330,0.032315
8,SKU100126,0.031160
9,SKU100113,0.031035


In [14]:
MIN_SUPPORT = 0.003

frequent_itemsets = fpgrowth(
    basket,
    min_support=MIN_SUPPORT,
    use_colnames=True
)

print("Frequent itemsets:", len(frequent_itemsets))

Frequent itemsets: 387


In [15]:
frequent_itemsets["itemset_size"] = (
    frequent_itemsets["itemsets"]
    .apply(len)
)

frequent_itemsets["itemset_size"].value_counts().sort_index()

itemset_size
1    311
2     58
3     18
Name: count, dtype: int64

In [16]:
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.05
)

print("Total rules:", len(rules))

Total rules: 224


In [17]:
product_lookup = (
    products[
        [
            "product_id",
            "product_name",
            "brand",
            "category",
            "subcategory",
            "unit_price_aed"
        ]
    ]
    .drop_duplicates("product_id")
    .set_index("product_id")
)

In [18]:
def itemset_to_names(itemset):
    names = []
    
    for product_id in itemset:
        if product_id in product_lookup.index:
            names.append(
                product_lookup.loc[
                    product_id,
                    "product_name"
                ]
            )
        else:
            names.append(str(product_id))
    
    return names

In [19]:
rules["antecedents_names"] = rules["antecedents"].apply(
    itemset_to_names
)

rules["consequents_names"] = rules["consequents"].apply(
    itemset_to_names
)

## Association Rule Metrics

### Support

Support measures how frequently the complete item combination appears
in all baskets.

Support(A → B) = P(A and B)

A rule with extremely low support may have a high lift simply because the
combination is rare. Such a rule may not be commercially useful.

### Confidence

Confidence measures how often B is purchased when A is purchased.

Confidence(A → B) = P(B | A)

Confidence is directional, so A → B and B → A can have different values.

### Lift

Lift compares the observed co-purchase rate with what would be expected
if A and B were independent.

Lift(A → B) = P(B | A) / P(B)

Lift > 1 indicates positive association.

### Leverage

Leverage measures the difference between observed joint support and the
support expected under independence.

Leverage = P(A and B) - P(A)P(B)

### Conviction

Conviction measures the degree to which A predicts the absence of B.

Higher conviction generally indicates a stronger directional dependency.

### Why a rule with lift 30 and support 0.02% can be useless

A very high lift does not automatically mean commercial value.

If a combination appears in only a tiny fraction of all orders, the rule
may have very little practical reach. A retailer could have an impressive
statistical association but almost no customers to whom the recommendation
can be shown.

Therefore rules should be filtered by support and confidence before being
ranked by lift.

In [20]:
MIN_RULE_SUPPORT = 0.003
MIN_CONFIDENCE = 0.10
MIN_LIFT = 1.20

meaningful_rules = rules[
    (rules["support"] >= MIN_RULE_SUPPORT) &
    (rules["confidence"] >= MIN_CONFIDENCE) &
    (rules["lift"] >= MIN_LIFT)
].copy()

print("Meaningful rules:", len(meaningful_rules))

Meaningful rules: 223


In [21]:
pair_rules = meaningful_rules[
    (meaningful_rules["antecedents"].apply(len) == 1) &
    (meaningful_rules["consequents"].apply(len) == 1)
].copy()

print("Pair rules:", len(pair_rules))

Pair rules: 116


In [22]:
top_rules = (
    pair_rules
    .sort_values(
        ["lift", "support"],
        ascending=[False, False]
    )
)

top_rules[
    [
        "antecedents_names",
        "consequents_names",
        "support",
        "confidence",
        "lift",
        "leverage",
        "conviction"
    ]
].head(20)

,antecedents_names,consequents_names,support,confidence,lift,leverage,conviction
170,[Vantar Kitchen Towels x6],[Aurelia Dish Soap 750ml],0.007326,0.789223,37.845144,0.007133,4.645422
171,[Aurelia Dish Soap 750ml],[Vantar Kitchen Towels x6],0.007326,0.351323,37.845144,0.007133,1.527288
208,[Meridian Essentials Stain Remover 500ml],[Peak & Pine Washing Powder 3kg],0.004127,0.779167,37.508432,0.004017,4.434235
209,[Peak & Pine Washing Powder 3kg],[Meridian Essentials Stain Remover 500ml],0.004127,0.198654,37.508432,0.004017,1.241292
98,[Orchard Lane Shampoo 400ml],[Meridian Essentials Conditioner 400ml],0.016448,0.640504,37.290508,0.016007,2.733895
99,[Meridian Essentials Conditioner 400ml],[Orchard Lane Shampoo 400ml],0.016448,0.957602,37.290508,0.016007,22.980185
222,[Meridian Essentials Penne 500g],[Zephyr Pesto 190g],0.004774,0.194195,36.363432,0.004643,1.234368
223,[Zephyr Pesto 190g],[Meridian Essentials Penne 500g],0.004774,0.893939,36.363432,0.004643,9.196784
133,[Sundara Ketchup 500ml],[Meridian Essentials Sliced Cheese x24],0.004038,0.399563,33.633830,0.003918,1.645669
132,[Meridian Essentials Sliced Cheese x24],[Sundara Ketchup 500ml],0.004038,0.339938,33.633830,0.003918,1.499697


In [23]:
rules_with_product_info = pair_rules.copy()

rules_with_product_info["antecedent_id"] = (
    rules_with_product_info["antecedents"]
    .apply(lambda x: list(x)[0])
)

rules_with_product_info["consequent_id"] = (
    rules_with_product_info["consequents"]
    .apply(lambda x: list(x)[0])
)

rules_with_product_info = (
    rules_with_product_info
    .merge(
        product_lookup.reset_index(),
        left_on="antecedent_id",
        right_on="product_id",
        how="left"
    )
    .rename(
        columns={
            "product_name": "antecedent_name",
            "category": "antecedent_category",
            "subcategory": "antecedent_subcategory"
        }
    )
)

rules_with_product_info = (
    rules_with_product_info
    .merge(
        product_lookup.reset_index(),
        left_on="consequent_id",
        right_on="product_id",
        how="left"
    )
    .rename(
        columns={
            "product_name": "consequent_name",
            "category": "consequent_category",
            "subcategory": "consequent_subcategory"
        }
    )
)

In [24]:
interesting_rules = rules_with_product_info[
    (
        rules_with_product_info["antecedent_subcategory"]
        !=
        rules_with_product_info["consequent_subcategory"]
    )
].copy()

interesting_rules = interesting_rules.sort_values(
    ["lift", "support"],
    ascending=[False, False]
)

interesting_rules[
    [
        "antecedent_name",
        "consequent_name",
        "antecedent_category",
        "consequent_category",
        "support",
        "confidence",
        "lift"
    ]
].head(20)


,antecedent_name,consequent_name,antecedent_category,consequent_category,support,confidence,lift
80,Vantar Kitchen Towels x6,Aurelia Dish Soap 750ml,Household,Household,0.007326,0.789223,37.845144
81,Aurelia Dish Soap 750ml,Vantar Kitchen Towels x6,Household,Household,0.007326,0.351323,37.845144
114,Meridian Essentials Penne 500g,Zephyr Pesto 190g,Grocery,Grocery,0.004774,0.194195,36.363432
115,Zephyr Pesto 190g,Meridian Essentials Penne 500g,Grocery,Grocery,0.004774,0.893939,36.363432
67,Sundara Ketchup 500ml,Meridian Essentials Sliced Cheese x24,Grocery,Fresh Food,0.004038,0.399563,33.633830
66,Meridian Essentials Sliced Cheese x24,Sundara Ketchup 500ml,Fresh Food,Grocery,0.004038,0.339938,33.633830
64,Meridian Essentials Beef Burger Patties x4,Sundara Ketchup 500ml,Fresh Food,Grocery,0.009379,0.327427,32.395952
65,Sundara Ketchup 500ml,Meridian Essentials Beef Burger Patties x4,Grocery,Fresh Food,0.009379,0.927948,32.395952
60,Meridian Essentials Beef Burger Patties x4,Meridian Essentials Sliced Cheese x24,Fresh Food,Fresh Food,0.010872,0.379558,31.949878
61,Meridian Essentials Sliced Cheese x24,Meridian Essentials Beef Burger Patties x4,Fresh Food,Fresh Food,0.010872,0.915170,31.949878


In [25]:
directional_rules = pair_rules.copy()

directional_rules["A"] = directional_rules[
    "antecedents"
].apply(lambda x: list(x)[0])

directional_rules["B"] = directional_rules[
    "consequents"
].apply(lambda x: list(x)[0]
)

In [26]:
directional_rules["pair"] = directional_rules.apply(
    lambda row: tuple(sorted([row["A"], row["B"]])),
    axis=1
)

In [27]:
direction_pairs = []

for pair, group in directional_rules.groupby("pair"):
    
    if len(group) == 2:
        rows = group.sort_values(
            "confidence",
            ascending=False
        )
        
        first = rows.iloc[0]
        second = rows.iloc[1]
        
        direction_pairs.append({
            "product_A": first["A"],
            "product_B": first["B"],
            "confidence_A_to_B": first["confidence"],
            "confidence_B_to_A": second["confidence"],
            "confidence_difference": abs(
                first["confidence"] -
                second["confidence"]
            ),
            "lift_A_to_B": first["lift"],
            "lift_B_to_A": second["lift"]
        })

direction_comparison = pd.DataFrame(direction_pairs)

direction_comparison = direction_comparison.sort_values(
    "confidence_difference",
    ascending=False
)

direction_comparison.head(10)

,product_A,product_B,confidence_A_to_B,confidence_B_to_A,confidence_difference,lift_A_to_B,lift_B_to_A
0,SKU100022,SKU100001,0.893939,0.194195,0.699744,36.363432,36.363432
4,SKU100015,SKU100145,0.927948,0.327427,0.600521,32.395952,32.395952
32,SKU100225,SKU100222,0.779167,0.198654,0.580512,37.508432,37.508432
11,SKU100057,SKU100065,0.641256,0.102417,0.538838,15.608862,15.608862
25,SKU100156,SKU100145,0.915170,0.379558,0.535612,31.949878,31.949878
15,SKU100189,SKU100065,0.758268,0.225783,0.532485,18.457076,18.457076
20,SKU100104,SKU100113,0.891846,0.373311,0.518535,28.737142,28.737142
8,SKU100024,SKU100162,0.829726,0.343215,0.486511,22.442713,22.442713
28,SKU100214,SKU100202,0.789223,0.351323,0.437901,37.845144,37.845144
13,SKU100101,SKU100059,0.902228,0.481670,0.420558,29.777476,29.777476


In [28]:
direction_comparison["product_A_name"] = (
    direction_comparison["product_A"]
    .map(product_lookup["product_name"])
)

direction_comparison["product_B_name"] = (
    direction_comparison["product_B"]
    .map(product_lookup["product_name"])
)

direction_comparison.head(10)

,product_A,product_B,confidence_A_to_B,confidence_B_to_A,confidence_difference,lift_A_to_B,lift_B_to_A,product_A_name,product_B_name
0,SKU100022,SKU100001,0.893939,0.194195,0.699744,36.363432,36.363432,Zephyr Pesto 190g,Meridian Essentials Penne 500g
4,SKU100015,SKU100145,0.927948,0.327427,0.600521,32.395952,32.395952,Sundara Ketchup 500ml,Meridian Essentials Beef Burger Patties x4
32,SKU100225,SKU100222,0.779167,0.198654,0.580512,37.508432,37.508432,Meridian Essentials Stain Remover 500ml,Peak & Pine Washing Powder 3kg
11,SKU100057,SKU100065,0.641256,0.102417,0.538838,15.608862,15.608862,Marisol Saffron 1g,Lumen & Co Turkish Coffee 200g
25,SKU100156,SKU100145,0.915170,0.379558,0.535612,31.949878,31.949878,Meridian Essentials Sliced Cheese x24,Meridian Essentials Beef Burger Patties x4
15,SKU100189,SKU100065,0.758268,0.225783,0.532485,18.457076,18.457076,Bright Harvest Ma'amoul 400g,Lumen & Co Turkish Coffee 200g
20,SKU100104,SKU100113,0.891846,0.373311,0.518535,28.737142,28.737142,Aurelia Butter 250g,Aurelia Sourdough Loaf
8,SKU100024,SKU100162,0.829726,0.343215,0.486511,22.442713,22.442713,Meridian Essentials Hummus Dip 250g,Nordvik Tortilla Chips 200g
28,SKU100214,SKU100202,0.789223,0.351323,0.437901,37.845144,37.845144,Vantar Kitchen Towels x6,Aurelia Dish Soap 750ml
13,SKU100101,SKU100059,0.902228,0.481670,0.420558,29.777476,29.777476,Sundara Full Fat Milk 2L,Peak & Pine Ground Coffee 250g


## Directionality Interpretation

Confidence is directional.

If A → B has substantially higher confidence than B → A, customers who buy
A are much more likely to also buy B than customers who buy B are to buy A.

This matters for merchandising placement.

For example, if A is a high-volume primary product and B is a complementary
product, a checkout prompt can be triggered after A is added to the basket.

The reverse recommendation may be much less effective.

In [30]:
transaction_category = (
    train_transactions[
        ["order_id", "product_id"]
    ]
    .drop_duplicates()
    .merge(
        products[
            ["product_id", "category"]
        ],
        on="product_id",
        how="left"
    )
    .dropna(subset=["category"])
    .drop_duplicates(
        subset=["order_id", "category"]
    )
)

In [31]:
category_basket = (
    transaction_category
    .assign(purchased=1)
    .pivot_table(
        index="order_id",
        columns="category",
        values="purchased",
        fill_value=0
    )
    .astype(bool)
)

category_basket.shape

(135945, 10)

In [32]:
category_itemsets = fpgrowth(
    category_basket,
    min_support=0.005,
    use_colnames=True
)

category_rules = association_rules(
    category_itemsets,
    metric="confidence",
    min_threshold=0.10
)

category_rules = category_rules[
    (category_rules["support"] >= 0.005) &
    (category_rules["confidence"] >= 0.10) &
    (category_rules["lift"] > 1)
].sort_values(
    "lift",
    ascending=False
)

category_rules.head(20)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
665,"frozenset({Home & Living, Household, Grocery})","frozenset({Pet Care, Baby})",0.032594,0.057553,0.005671,0.174001,3.023340,1.0,0.003796,1.140979,0.691788,0.067137,0.123560,0.136272
663,"frozenset({Home & Living, Pet Care, Grocery})","frozenset({Household, Baby})",0.022399,0.084939,0.005671,0.253202,2.980994,1.0,0.003769,1.225313,0.679767,0.055785,0.183882,0.159986
662,"frozenset({Home & Living, Pet Care, Household})","frozenset({Grocery, Baby})",0.021972,0.087969,0.005671,0.258119,2.934185,1.0,0.003739,1.229348,0.673999,0.054392,0.186561,0.161294
1156,"frozenset({Pet Care, Baby})","frozenset({Grocery, Beverages, Household})",0.057553,0.035566,0.005885,0.102249,2.874934,1.0,0.003838,1.074279,0.691992,0.067459,0.069143,0.133855
1149,"frozenset({Grocery, Beverages, Household})","frozenset({Pet Care, Baby})",0.035566,0.057553,0.005885,0.165460,2.874934,1.0,0.003838,1.129302,0.676216,0.067459,0.114497,0.133855
669,"frozenset({Household, Pet Care, Baby})","frozenset({Home & Living, Grocery})",0.026319,0.075097,0.005671,0.215484,2.869420,1.0,0.003695,1.178947,0.669108,0.059235,0.151786,0.145503
1028,"frozenset({Snacks, Household, Baby})","frozenset({Grocery, Pet Care})",0.029343,0.070948,0.005944,0.202557,2.855015,1.0,0.003862,1.165039,0.669381,0.062997,0.141660,0.143166
1150,"frozenset({Grocery, Beverages, Baby})","frozenset({Household, Pet Care})",0.030373,0.068256,0.005885,0.193752,2.838619,1.0,0.003812,1.155654,0.668005,0.063452,0.134689,0.139984
1151,"frozenset({Household, Beverages, Baby})","frozenset({Grocery, Pet Care})",0.029232,0.070948,0.005885,0.201309,2.837417,1.0,0.003811,1.163218,0.667067,0.062407,0.140316,0.142127
1029,"frozenset({Snacks, Grocery, Baby})","frozenset({Household, Pet Care})",0.030748,0.068256,0.005944,0.193301,2.832025,1.0,0.003845,1.155009,0.667417,0.063868,0.134206,0.140190


In [33]:
category_rules["antecedent"] = (
    category_rules["antecedents"]
    .apply(lambda x: ", ".join(sorted(x)))
)

category_rules["consequent"] = (
    category_rules["consequents"]
    .apply(lambda x: ", ".join(sorted(x)))
)

category_rules[
    [
        "antecedent",
        "consequent",
        "support",
        "confidence",
        "lift"
    ]
].head(20)

,antecedent,consequent,support,confidence,lift
665,"Grocery, Home & Living, Household","Baby, Pet Care",0.005671,0.174001,3.023340
663,"Grocery, Home & Living, Pet Care","Baby, Household",0.005671,0.253202,2.980994
662,"Home & Living, Household, Pet Care","Baby, Grocery",0.005671,0.258119,2.934185
1156,"Baby, Pet Care","Beverages, Grocery, Household",0.005885,0.102249,2.874934
1149,"Beverages, Grocery, Household","Baby, Pet Care",0.005885,0.165460,2.874934
669,"Baby, Household, Pet Care","Grocery, Home & Living",0.005671,0.215484,2.869420
1028,"Baby, Household, Snacks","Grocery, Pet Care",0.005944,0.202557,2.855015
1150,"Baby, Beverages, Grocery","Household, Pet Care",0.005885,0.193752,2.838619
1151,"Baby, Beverages, Household","Grocery, Pet Care",0.005885,0.201309,2.837417
1029,"Baby, Grocery, Snacks","Household, Pet Care",0.005944,0.193301,2.832025


In [34]:
transaction_subcategory = (
    train_transactions[
        ["order_id", "product_id"]
    ]
    .drop_duplicates()
    .merge(
        products[
            ["product_id", "subcategory"]
        ],
        on="product_id",
        how="left"
    )
    .dropna(subset=["subcategory"])
    .drop_duplicates(
        subset=["order_id", "subcategory"]
    )
)

In [35]:
subcategory_basket = (
    transaction_subcategory
    .assign(purchased=1)
    .pivot_table(
        index="order_id",
        columns="subcategory",
        values="purchased",
        fill_value=0
    )
    .astype(bool)
)

In [36]:
subcategory_itemsets = fpgrowth(
    subcategory_basket,
    min_support=0.005,
    use_colnames=True
)

subcategory_rules = association_rules(
    subcategory_itemsets,
    metric="confidence",
    min_threshold=0.10
)

subcategory_rules = subcategory_rules[
    (subcategory_rules["support"] >= 0.005) &
    (subcategory_rules["confidence"] >= 0.10) &
    (subcategory_rules["lift"] > 1)
].sort_values(
    "lift",
    ascending=False
)

subcategory_rules.head(20)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
268,"frozenset({Diapers, Baby Care})",frozenset({Formula & Milk}),0.018176,0.079370,0.010004,0.550384,6.934385,1.0,0.008561,2.047593,0.871634,0.114276,0.511622,0.338214
271,frozenset({Formula & Milk}),"frozenset({Diapers, Baby Care})",0.079370,0.018176,0.010004,0.126043,6.934385,1.0,0.008561,1.123423,0.929572,0.114276,0.109863,0.338214
388,"frozenset({Cheese & Deli, Canned & Jarred})",frozenset({Pasta & Rice}),0.010033,0.105403,0.006709,0.668622,6.343484,1.0,0.005651,2.699625,0.850895,0.061701,0.629578,0.366134
267,"frozenset({Formula & Milk, Baby Care})",frozenset({Diapers}),0.023009,0.070315,0.010004,0.434783,6.183337,1.0,0.008386,1.644827,0.858017,0.120067,0.392033,0.288528
272,frozenset({Diapers}),"frozenset({Formula & Milk, Baby Care})",0.070315,0.023009,0.010004,0.142274,6.183337,1.0,0.008386,1.139048,0.901677,0.120067,0.122074,0.288528
270,frozenset({Baby Care}),"frozenset({Formula & Milk, Diapers})",0.082776,0.019905,0.010004,0.120857,6.071640,1.0,0.008356,1.114829,0.910683,0.107945,0.103002,0.311722
269,"frozenset({Formula & Milk, Diapers})",frozenset({Baby Care}),0.019905,0.082776,0.010004,0.502587,6.071640,1.0,0.008356,1.843988,0.852264,0.107945,0.457697,0.311722
184,"frozenset({Cheese & Deli, Bakery})",frozenset({Meat & Poultry}),0.010990,0.115422,0.007312,0.665328,5.764324,1.0,0.006043,2.643120,0.835703,0.061392,0.621659,0.364338
389,"frozenset({Cheese & Deli, Pasta & Rice})",frozenset({Canned & Jarred}),0.016521,0.071271,0.006709,0.406055,5.697304,1.0,0.005531,1.563661,0.838329,0.082736,0.360475,0.250091
220,"frozenset({Sauces & Condiments, Bakery})",frozenset({Meat & Poultry}),0.009570,0.115422,0.006113,0.638739,5.533964,1.0,0.005008,2.448588,0.827214,0.051420,0.591601,0.345850


In [37]:
subcategory_rules["antecedent"] = (
    subcategory_rules["antecedents"]
    .apply(lambda x: ", ".join(sorted(x)))
)

subcategory_rules["consequent"] = (
    subcategory_rules["consequents"]
    .apply(lambda x: ", ".join(sorted(x)))
)

subcategory_rules[
    [
        "antecedent",
        "consequent",
        "support",
        "confidence",
        "lift"
    ]
].head(20)

,antecedent,consequent,support,confidence,lift
268,"Baby Care, Diapers",Formula & Milk,0.010004,0.550384,6.934385
271,Formula & Milk,"Baby Care, Diapers",0.010004,0.126043,6.934385
388,"Canned & Jarred, Cheese & Deli",Pasta & Rice,0.006709,0.668622,6.343484
267,"Baby Care, Formula & Milk",Diapers,0.010004,0.434783,6.183337
272,Diapers,"Baby Care, Formula & Milk",0.010004,0.142274,6.183337
270,Baby Care,"Diapers, Formula & Milk",0.010004,0.120857,6.071640
269,"Diapers, Formula & Milk",Baby Care,0.010004,0.502587,6.071640
184,"Bakery, Cheese & Deli",Meat & Poultry,0.007312,0.665328,5.764324
389,"Cheese & Deli, Pasta & Rice",Canned & Jarred,0.006709,0.406055,5.697304
220,"Bakery, Sauces & Condiments",Meat & Poultry,0.006113,0.638739,5.533964


In [38]:
customer_segments.columns

Index(['customer_id', 'cluster', 'segment_name'], dtype='str')

In [39]:
segment_column = "segment_name"

print(
    customer_segments[
        [segment_column]
    ].value_counts()
)

segment_name                   
Premium Home & Living Shoppers     3025
Frequent Personal Care Shoppers    2600
Family & Household Shoppers        2389
Health & Wellness Specialists      1631
Lapsed Everyday Shoppers           1320
Name: count, dtype: int64


In [40]:
segmented_transactions = train_transactions.merge(
    customer_segments[
        ["customer_id", segment_column]
    ],
    on="customer_id",
    how="inner"
)

print(
    segmented_transactions[segment_column]
    .value_counts()
)

segment_name
Premium Home & Living Shoppers     219932
Frequent Personal Care Shoppers    137751
Family & Household Shoppers         72166
Health & Wellness Specialists       54122
Lapsed Everyday Shoppers            10975
Name: count, dtype: int64


In [41]:
def get_segment_rules(
    segment_name,
    min_support=0.003,
    min_confidence=0.10,
    min_lift=1.20
):
    
    segment_tx = segmented_transactions[
        segmented_transactions[segment_column] == segment_name
    ].copy()
    
    segment_basket_tx = (
        segment_tx[
            ["order_id", "product_id"]
        ]
        .drop_duplicates()
    )
    
    segment_basket = (
        segment_basket_tx
        .assign(purchased=1)
        .pivot_table(
            index="order_id",
            columns="product_id",
            values="purchased",
            fill_value=0
        )
        .astype(bool)
    )
    
    if segment_basket.shape[1] < 2:
        return pd.DataFrame()
    
    itemsets = fpgrowth(
        segment_basket,
        min_support=min_support,
        use_colnames=True
    )
    
    if itemsets.empty:
        return pd.DataFrame()
    
    segment_rules = association_rules(
        itemsets,
        metric="confidence",
        min_threshold=min_confidence
    )
    
    segment_rules = segment_rules[
        (segment_rules["support"] >= min_support) &
        (segment_rules["confidence"] >= min_confidence) &
        (segment_rules["lift"] >= min_lift)
    ].copy()
    
    if segment_rules.empty:
        return segment_rules
    
    segment_rules["antecedents_names"] = (
        segment_rules["antecedents"]
        .apply(itemset_to_names)
    )
    
    segment_rules["consequents_names"] = (
        segment_rules["consequents"]
        .apply(itemset_to_names)
    )
    
    return segment_rules.sort_values(
        ["lift", "support"],
        ascending=[False, False]
    )

In [42]:
segment_rule_results = {}

for segment in sorted(
    segmented_transactions[segment_column].dropna().unique()
):
    
    print("\n", "=" * 60)
    print(segment)
    
    segment_rules = get_segment_rules(segment)
    
    segment_rule_results[segment] = segment_rules
    
    print("Rules:", len(segment_rules))
    
    if len(segment_rules) > 0:
        display(
            segment_rules[
                [
                    "antecedents_names",
                    "consequents_names",
                    "support",
                    "confidence",
                    "lift"
                ]
            ].head(10)
        )


Family & Household Shoppers
Rules: 224


,antecedents_names,consequents_names,support,confidence,lift
250,[Meridian Essentials Conditioner 400ml],[Orchard Lane Shampoo 400ml],0.010453,0.921053,55.266459
249,[Orchard Lane Shampoo 400ml],[Meridian Essentials Conditioner 400ml],0.010453,0.627240,55.266459
222,"[Meridian Essentials Beef Burger Patties x4, B...",[Sundara Ketchup 500ml],0.004958,0.417085,45.340436
227,[Sundara Ketchup 500ml],"[Meridian Essentials Beef Burger Patties x4, B...",0.004958,0.538961,45.340436
148,[Lumen & Co Toothbrush x2],[Halcyon Toothpaste 100ml],0.006511,0.664634,43.805670
149,[Halcyon Toothpaste 100ml],[Lumen & Co Toothbrush x2],0.006511,0.429134,43.805670
64,"[Peak & Pine Stage 1 Formula 900g, Zephyr Napp...","[Sundara Newborn Diapers x64, Sundara Baby Wip...",0.004301,0.510638,43.615284
59,"[Sundara Newborn Diapers x64, Sundara Baby Wip...","[Peak & Pine Stage 1 Formula 900g, Zephyr Napp...",0.004301,0.367347,43.615284
61,"[Sundara Newborn Diapers x64, Peak & Pine Stag...","[Sundara Baby Wipes x144, Zephyr Nappy Rash Cr...",0.004301,0.269663,43.407952
62,"[Sundara Baby Wipes x144, Zephyr Nappy Rash Cr...","[Sundara Newborn Diapers x64, Peak & Pine Stag...",0.004301,0.692308,43.407952



Frequent Personal Care Shoppers
Rules: 189


,antecedents_names,consequents_names,support,confidence,lift
165,[Vantar Kitchen Towels x6],[Aurelia Dish Soap 750ml],0.004527,0.838488,67.456983
166,[Aurelia Dish Soap 750ml],[Vantar Kitchen Towels x6],0.004527,0.364179,67.456983
99,[Meridian Essentials Penne 500g],"[Little Otter Chopped Tomatoes 400g, Solmar Pa...",0.004211,0.255056,60.298413
98,"[Little Otter Chopped Tomatoes 400g, Solmar Pa...",[Meridian Essentials Penne 500g],0.004211,0.995614,60.298413
185,[Meridian Essentials Penne 500g],[Zephyr Pesto 190g],0.003432,0.207865,56.587618
186,[Zephyr Pesto 190g],[Meridian Essentials Penne 500g],0.003432,0.934343,56.587618
126,"[Meridian Essentials Beef Burger Patties x4, M...",[Sundara Ketchup 500ml],0.003172,0.366167,51.939829
131,[Sundara Ketchup 500ml],"[Meridian Essentials Beef Burger Patties x4, M...",0.003172,0.450000,51.939829
96,"[Meridian Essentials Penne 500g, Little Otter ...",[Solmar Parmesan Wedge 200g],0.004211,0.457661,51.608491
101,[Solmar Parmesan Wedge 200g],"[Meridian Essentials Penne 500g, Little Otter ...",0.004211,0.474895,51.608491



Health & Wellness Specialists
Rules: 175


,antecedents_names,consequents_names,support,confidence,lift
180,[Meridian Essentials Hummus Dip 250g],[Nordvik Tortilla Chips 200g],0.003218,0.643678,70.004023
181,[Nordvik Tortilla Chips 200g],[Meridian Essentials Hummus Dip 250g],0.003218,0.350000,70.004023
178,[Vantar Kitchen Towels x6],[Aurelia Dish Soap 750ml],0.003678,0.810127,66.183158
179,[Aurelia Dish Soap 750ml],[Vantar Kitchen Towels x6],0.003678,0.300469,66.183158
172,"[Little Otter Chopped Tomatoes 400g, Solmar Pa...",[Meridian Essentials Penne 500g],0.003735,1.000000,66.163498
173,[Meridian Essentials Penne 500g],"[Little Otter Chopped Tomatoes 400g, Solmar Pa...",0.003735,0.247148,66.163498
183,[Zephyr Pesto 190g],[Meridian Essentials Penne 500g],0.003046,0.929825,61.520446
182,[Meridian Essentials Penne 500g],[Zephyr Pesto 190g],0.003046,0.201521,61.520446
55,[Kaviro Sponge Pack x6],[Aurelia Dish Soap 750ml],0.006724,0.606218,49.524849
56,[Aurelia Dish Soap 750ml],[Kaviro Sponge Pack x6],0.006724,0.549296,49.524849



Lapsed Everyday Shoppers
Rules: 180


,antecedents_names,consequents_names,support,confidence,lift
164,[Orchard Lane Shampoo 400ml],[Meridian Essentials Conditioner 400ml],0.007649,0.465517,58.688424
165,[Meridian Essentials Conditioner 400ml],[Orchard Lane Shampoo 400ml],0.007649,0.964286,58.688424
137,[Halcyon Toothpaste 100ml],[Lumen & Co Toothbrush x2],0.004816,0.377778,55.564815
136,[Lumen & Co Toothbrush x2],[Halcyon Toothpaste 100ml],0.004816,0.708333,55.564815
24,"[Peak & Pine Ground Coffee 250g, Sundara Diges...",[Sundara Full Fat Milk 2L],0.003966,0.608696,39.067194
25,[Sundara Full Fat Milk 2L],"[Peak & Pine Ground Coffee 250g, Sundara Diges...",0.003966,0.254545,39.067194
23,"[Sundara Full Fat Milk 2L, Sundara Digestive B...",[Peak & Pine Ground Coffee 250g],0.003966,1.000000,37.956989
5,"[Sundara Full Fat Milk 2L, Peak & Pine Coffee ...",[Peak & Pine Ground Coffee 250g],0.003116,1.000000,37.956989
26,[Peak & Pine Ground Coffee 250g],"[Sundara Full Fat Milk 2L, Sundara Digestive B...",0.003966,0.150538,37.956989
8,[Peak & Pine Ground Coffee 250g],"[Sundara Full Fat Milk 2L, Peak & Pine Coffee ...",0.003116,0.118280,37.956989



Premium Home & Living Shoppers
Rules: 222


,antecedents_names,consequents_names,support,confidence,lift
95,"[Aurelia Hair Oil 200ml, Meridian Essentials C...",[Orchard Lane Shampoo 400ml],0.003719,0.993976,46.181890
98,[Orchard Lane Shampoo 400ml],"[Aurelia Hair Oil 200ml, Meridian Essentials C...",0.003719,0.172775,46.181890
90,[Orchard Lane Shampoo 400ml],[Meridian Essentials Conditioner 400ml],0.013793,0.640838,44.290669
91,[Meridian Essentials Conditioner 400ml],[Orchard Lane Shampoo 400ml],0.013793,0.953271,44.290669
94,"[Aurelia Hair Oil 200ml, Orchard Lane Shampoo ...",[Meridian Essentials Conditioner 400ml],0.003719,0.637066,44.029968
99,[Meridian Essentials Conditioner 400ml],"[Aurelia Hair Oil 200ml, Orchard Lane Shampoo ...",0.003719,0.257009,44.029968
175,[Casa Verde Shaker Bottle 700ml],[Aurelia Whey Protein 2kg],0.005116,0.602122,43.091542
176,[Aurelia Whey Protein 2kg],[Casa Verde Shaker Bottle 700ml],0.005116,0.366129,43.091542
157,"[Kaviro Sponge Pack x6, Vantar Kitchen Towels x6]",[Aurelia Dish Soap 750ml],0.004327,0.989691,41.902259
162,[Aurelia Dish Soap 750ml],"[Kaviro Sponge Pack x6, Vantar Kitchen Towels x6]",0.004327,0.183206,41.902259


In [43]:
customer_product = (
    train_transactions[
        ["customer_id", "product_id"]
    ]
    .drop_duplicates()
    .assign(purchased=1)
)

In [44]:
customer_product_matrix = (
    customer_product
    .pivot_table(
        index="customer_id",
        columns="product_id",
        values="purchased",
        fill_value=0
    )
)

print(
    "Customer-product matrix:",
    customer_product_matrix.shape
)

Customer-product matrix: (10373, 391)


In [45]:
item_customer_matrix = customer_product_matrix.T

item_similarity = cosine_similarity(
    item_customer_matrix
)

item_similarity_df = pd.DataFrame(
    item_similarity,
    index=item_customer_matrix.index,
    columns=item_customer_matrix.index
)

print("Similarity matrix:", item_similarity_df.shape)

Similarity matrix: (391, 391)


In [46]:
def get_similar_products(
    product_id,
    top_n=10
):
    
    if product_id not in item_similarity_df.index:
        return pd.DataFrame()
    
    scores = (
        item_similarity_df[product_id]
        .drop(index=product_id)
        .sort_values(ascending=False)
        .head(top_n)
    )
    
    result = scores.reset_index()
    
    result.columns = [
        "product_id",
        "similarity"
    ]
    
    result["product_name"] = (
        result["product_id"]
        .map(product_lookup["product_name"])
    )
    
    result["category"] = (
        result["product_id"]
        .map(product_lookup["category"])
    )
    
    return result

In [47]:
example_product = customer_product_matrix.columns[0]

get_similar_products(
    example_product,
    top_n=10
)

,product_id,similarity,product_name,category
0,SKU100041,0.706939,Little Otter Chopped Tomatoes 400g,Grocery
1,SKU100153,0.621963,Solmar Parmesan Wedge 200g,Fresh Food
2,SKU100022,0.476740,Zephyr Pesto 190g,Grocery
3,SKU100376,0.424456,Meridian Essentials Dry Dog Food 3kg,Pet Care
4,SKU100218,0.397546,Meridian Essentials Baking Paper 30m,Household
5,SKU100162,0.397425,Nordvik Tortilla Chips 200g,Snacks
6,SKU100065,0.395688,Lumen & Co Turkish Coffee 200g,Beverages
7,SKU100126,0.389653,Terra Nova Tomatoes 1kg,Fresh Food
8,SKU100077,0.389209,Meridian Essentials Lemon Soda 1.25L,Beverages
9,SKU100378,0.387836,Nordvik Dry Cat Food 2kg,Pet Care


In [48]:
sparse_matrix = csr_matrix(
    customer_product_matrix.values
)

n_components = min(
    20,
    min(sparse_matrix.shape) - 1
)

svd = TruncatedSVD(
    n_components=n_components,
    random_state=42
)

customer_latent = svd.fit_transform(
    sparse_matrix
)

product_latent = svd.components_.T

print("Latent dimensions:", n_components)

Latent dimensions: 20


In [49]:
svd_product_similarity = cosine_similarity(
    product_latent
)

svd_similarity_df = pd.DataFrame(
    svd_product_similarity,
    index=customer_product_matrix.columns,
    columns=customer_product_matrix.columns
)

In [50]:
def get_svd_similar_products(
    product_id,
    top_n=10
):
    
    if product_id not in svd_similarity_df.index:
        return pd.DataFrame()
    
    scores = (
        svd_similarity_df[product_id]
        .drop(index=product_id)
        .sort_values(ascending=False)
        .head(top_n)
    )
    
    result = scores.reset_index()
    
    result.columns = [
        "product_id",
        "similarity"
    ]
    
    result["product_name"] = (
        result["product_id"]
        .map(product_lookup["product_name"])
    )
    
    result["category"] = (
        result["product_id"]
        .map(product_lookup["category"])
    )
    
    return result

In [51]:
product_popularity = (
    train_transactions
    .groupby("product_id")["order_id"]
    .nunique()
    .sort_values(ascending=False)
)

popular_products = (
    product_popularity
    .index
    .tolist()
)

popular_products[:10]

['SKU100376',
 'SKU100065',
 'SKU100162',
 'SKU100077',
 'SKU100369',
 'SKU100218',
 'SKU100277',
 'SKU100330',
 'SKU100126',
 'SKU100113']

In [52]:
popular_product_table = pd.DataFrame({
    "product_id": popular_products[:20]
})

popular_product_table["product_name"] = (
    popular_product_table["product_id"]
    .map(product_lookup["product_name"])
)

popular_product_table["orders"] = (
    popular_product_table["product_id"]
    .map(product_popularity)
)

popular_product_table

,product_id,product_name,orders
0,SKU100376,Meridian Essentials Dry Dog Food 3kg,5742
1,SKU100065,Lumen & Co Turkish Coffee 200g,5585
2,SKU100162,Nordvik Tortilla Chips 200g,5026
3,SKU100077,Meridian Essentials Lemon Soda 1.25L,4863
4,SKU100369,Peak & Pine Air Fryer 5L,4759
5,SKU100218,Meridian Essentials Baking Paper 30m,4675
6,SKU100277,Sundara Scented Candle 220g,4431
7,SKU100330,Vantar Electrolyte Tabs x20,4393
8,SKU100126,Terra Nova Tomatoes 1kg,4236
9,SKU100113,Aurelia Sourdough Loaf,4219


In [53]:
def recommend_for_customer(
    customer_id,
    similarity_df=item_similarity_df,
    top_n=10
):
    
    if customer_id not in customer_product_matrix.index:
        return pd.DataFrame()
    
    purchased = (
        customer_product_matrix
        .loc[customer_id]
    )
    
    purchased_products = purchased[
        purchased > 0
    ].index.tolist()
    
    if len(purchased_products) == 0:
        return pd.DataFrame()
    
    scores = pd.Series(
        0.0,
        index=similarity_df.index
    )
    
    for product_id in purchased_products:
        
        if product_id in similarity_df.index:
            scores = scores.add(
                similarity_df[product_id],
                fill_value=0
            )
    
    scores = scores.drop(
        labels=purchased_products,
        errors="ignore"
    )
    
    scores = scores.sort_values(
        ascending=False
    ).head(top_n)
    
    result = scores.reset_index()
    
    result.columns = [
        "product_id",
        "recommendation_score"
    ]
    
    result["product_name"] = (
        result["product_id"]
        .map(product_lookup["product_name"])
    )
    
    result["category"] = (
        result["product_id"]
        .map(product_lookup["category"])
    )
    
    return result

In [54]:
example_customer = customer_product_matrix.index[0]

recommend_for_customer(
    example_customer,
    top_n=10
)

,product_id,recommendation_score,product_name,category
0,SKU100065,6.364947,Lumen & Co Turkish Coffee 200g,Beverages
1,SKU100330,6.290101,Vantar Electrolyte Tabs x20,Health & Wellness
2,SKU100113,6.029791,Aurelia Sourdough Loaf,Fresh Food
3,SKU100126,5.918897,Terra Nova Tomatoes 1kg,Fresh Food
4,SKU100324,5.875679,Casa Verde Protein Bars x12,Health & Wellness
5,SKU100145,5.804380,Meridian Essentials Beef Burger Patties x4,Fresh Food
6,SKU100378,5.747432,Nordvik Dry Cat Food 2kg,Pet Care
7,SKU100059,5.664565,Peak & Pine Ground Coffee 250g,Beverages
8,SKU100369,5.634112,Peak & Pine Air Fryer 5L,Home & Living
9,SKU100241,5.608378,Orchard Lane Shampoo 400ml,Personal Care


In [55]:
test_customer_products = (
    test_transactions[
        ["customer_id", "product_id"]
    ]
    .drop_duplicates()
    .groupby("customer_id")["product_id"]
    .apply(set)
)

In [56]:
evaluation_customers = [
    customer_id
    for customer_id in test_customer_products.index
    if customer_id in customer_product_matrix.index
]

print(
    "Customers available for recommendation evaluation:",
    len(evaluation_customers)
)

Customers available for recommendation evaluation: 5832


In [57]:
def precision_at_k(recommended, actual, k):
    
    recommended_k = set(recommended[:k])
    
    if len(recommended_k) == 0:
        return 0.0
    
    return len(
        recommended_k.intersection(actual)
    ) / len(recommended_k)

In [58]:
def recall_at_k(recommended, actual, k):
    
    if len(actual) == 0:
        return 0.0
    
    recommended_k = set(recommended[:k])
    
    return len(
        recommended_k.intersection(actual)
    ) / len(actual)

In [59]:
def hit_rate_at_k(recommended, actual, k):
    
    recommended_k = set(recommended[:k])
    
    return int(
        len(recommended_k.intersection(actual)) > 0
    )

In [60]:
item_results = []

for customer_id in evaluation_customers:
    
    actual_products = test_customer_products[
        customer_id
    ]
    
    recommendations = recommend_for_customer(
        customer_id,
        similarity_df=item_similarity_df,
        top_n=10
    )
    
    recommended_products = (
        recommendations["product_id"]
        .tolist()
        if not recommendations.empty
        else []
    )
    
    item_results.append({
        "customer_id": customer_id,
        "precision_at_5": precision_at_k(
            recommended_products,
            actual_products,
            5
        ),
        "recall_at_10": recall_at_k(
            recommended_products,
            actual_products,
            10
        ),
        "hit_rate_at_5": hit_rate_at_k(
            recommended_products,
            actual_products,
            5
        )
    })

item_evaluation = pd.DataFrame(
    item_results
)

item_evaluation.mean(numeric_only=True)

precision_at_5    0.116598
recall_at_10      0.092058
hit_rate_at_5     0.433128
dtype: float64

In [61]:
customer_similarity = cosine_similarity(
    customer_latent
)

customer_similarity_df = pd.DataFrame(
    customer_similarity,
    index=customer_product_matrix.index,
    columns=customer_product_matrix.index
)

In [62]:
def svd_recommend_for_customer(
    customer_id,
    top_n=10
):
    
    if customer_id not in customer_product_matrix.index:
        return pd.DataFrame()
    
    customer_idx = (
        customer_product_matrix.index
        .get_loc(customer_id)
    )
    
    scores = (
        customer_latent[customer_idx]
        @ product_latent.T
    )
    
    score_series = pd.Series(
        scores,
        index=customer_product_matrix.columns
    )
    
    purchased = (
        customer_product_matrix
        .loc[customer_id]
    )
    
    purchased_products = purchased[
        purchased > 0
    ].index
    
    score_series = score_series.drop(
        labels=purchased_products,
        errors="ignore"
    )
    
    score_series = score_series.sort_values(
        ascending=False
    ).head(top_n)
    
    result = score_series.reset_index()
    
    result.columns = [
        "product_id",
        "recommendation_score"
    ]
    
    result["product_name"] = (
        result["product_id"]
        .map(product_lookup["product_name"])
    )
    
    result["category"] = (
        result["product_id"]
        .map(product_lookup["category"])
    )
    
    return result

In [63]:
svd_results = []

for customer_id in evaluation_customers:
    
    actual_products = test_customer_products[
        customer_id
    ]
    
    recommendations = svd_recommend_for_customer(
        customer_id,
        top_n=10
    )
    
    recommended_products = (
        recommendations["product_id"].tolist()
        if not recommendations.empty
        else []
    )
    
    svd_results.append({
        "customer_id": customer_id,
        "precision_at_5": precision_at_k(
            recommended_products,
            actual_products,
            5
        ),
        "recall_at_10": recall_at_k(
            recommended_products,
            actual_products,
            10
        ),
        "hit_rate_at_5": hit_rate_at_k(
            recommended_products,
            actual_products,
            5
        )
    })

svd_evaluation = pd.DataFrame(
    svd_results
)

svd_evaluation.mean(numeric_only=True)

precision_at_5    0.104458
recall_at_10      0.084558
hit_rate_at_5     0.400892
dtype: float64

In [64]:
baseline_results = []

for customer_id in evaluation_customers:
    
    actual_products = test_customer_products[
        customer_id
    ]
    
    # Recommend the same top sellers to everyone
    recommendations = popular_products[:10]
    
    baseline_results.append({
        "customer_id": customer_id,
        "precision_at_5": precision_at_k(
            recommendations,
            actual_products,
            5
        ),
        "recall_at_10": recall_at_k(
            recommendations,
            actual_products,
            10
        ),
        "hit_rate_at_5": hit_rate_at_k(
            recommendations,
            actual_products,
            5
        )
    })

baseline_evaluation = pd.DataFrame(
    baseline_results
)

baseline_evaluation.mean(numeric_only=True)

precision_at_5    0.118690
recall_at_10      0.097602
hit_rate_at_5     0.441701
dtype: float64

In [65]:
recommendation_comparison = pd.DataFrame({
    "model": [
        "Popularity Baseline",
        "Item-Item Cosine",
        "Truncated SVD"
    ],
    
    "precision_at_5": [
        baseline_evaluation["precision_at_5"].mean(),
        item_evaluation["precision_at_5"].mean(),
        svd_evaluation["precision_at_5"].mean()
    ],
    
    "recall_at_10": [
        baseline_evaluation["recall_at_10"].mean(),
        item_evaluation["recall_at_10"].mean(),
        svd_evaluation["recall_at_10"].mean()
    ],
    
    "hit_rate_at_5": [
        baseline_evaluation["hit_rate_at_5"].mean(),
        item_evaluation["hit_rate_at_5"].mean(),
        svd_evaluation["hit_rate_at_5"].mean()
    ]
})

recommendation_comparison

,model,precision_at_5,recall_at_10,hit_rate_at_5
0,Popularity Baseline,0.118690,0.097602,0.441701
1,Item-Item Cosine,0.116598,0.092058,0.433128
2,Truncated SVD,0.104458,0.084558,0.400892


In [66]:
baseline_precision = (
    recommendation_comparison
    .loc[
        recommendation_comparison["model"]
        == "Popularity Baseline",
        "precision_at_5"
    ]
    .iloc[0]
)

best_model_row = (
    recommendation_comparison
    .sort_values(
        "precision_at_5",
        ascending=False
    )
    .iloc[0]
)

print(
    "Popularity Precision@5:",
    round(baseline_precision, 4)
)

print(
    "Best model:",
    best_model_row["model"]
)

print(
    "Best Precision@5:",
    round(
        best_model_row["precision_at_5"],
        4
    )
)

Popularity Precision@5: 0.1187
Best model: Popularity Baseline
Best Precision@5: 0.1187


In [67]:
top_20_products = set(
    popular_products[:20]
)

In [68]:
def bestseller_overlap(
    recommendation_df,
    top_products
):
    
    if recommendation_df.empty:
        return 0
    
    recommended = recommendation_df[
        "product_id"
    ].tolist()
    
    return len(
        set(recommended).intersection(top_products)
    ) / len(recommended)

In [69]:
example_recommendations = recommend_for_customer(
    example_customer,
    top_n=10
)

overlap = bestseller_overlap(
    example_recommendations,
    top_20_products
)

print(
    "Bestseller overlap:",
    f"{overlap:.2%}"
)

Bestseller overlap: 90.00%


## Bestseller Bias

A recommender with very high overlap with the most popular products may be
reproducing popularity rather than discovering meaningful personalized
affinities.

Bestseller overlap should therefore be monitored alongside recommendation
accuracy.

Potential corrections include:

- Popularity penalties
- Re-ranking recommendations
- Category diversification
- Segment-aware recommendations
- Minimum novelty requirements

In [70]:
def cold_start_recommendation(
    customer_id=None,
    product_id=None,
    top_n=10
):
    
    # New customer
    if (
        customer_id is None or
        customer_id not in customer_product_matrix.index
    ):
        return popular_product_table.head(top_n)
    
    # New / unseen product
    if (
        product_id is not None and
        product_id not in item_similarity_df.index
    ):
        return popular_product_table.head(top_n)
    
    # Existing customer with history
    return recommend_for_customer(
        customer_id,
        top_n=top_n
    )

In [71]:
cold_start_recommendation(
    customer_id="NEW_CUSTOMER",
    top_n=10
)

,product_id,product_name,orders
0,SKU100376,Meridian Essentials Dry Dog Food 3kg,5742
1,SKU100065,Lumen & Co Turkish Coffee 200g,5585
2,SKU100162,Nordvik Tortilla Chips 200g,5026
3,SKU100077,Meridian Essentials Lemon Soda 1.25L,4863
4,SKU100369,Peak & Pine Air Fryer 5L,4759
5,SKU100218,Meridian Essentials Baking Paper 30m,4675
6,SKU100277,Sundara Scented Candle 220g,4431
7,SKU100330,Vantar Electrolyte Tabs x20,4393
8,SKU100126,Terra Nova Tomatoes 1kg,4236
9,SKU100113,Aurelia Sourdough Loaf,4219


In [72]:
bundle_rules = interesting_rules.copy()

bundle_rules["orders_with_rule"] = (
    bundle_rules["support"] *
    basket.shape[0]
)

bundle_rules["antecedent_orders"] = (
    basket[list(bundle_rules.iloc[0]["antecedents"])]
)

In [73]:
bundle_rules["estimated_target_orders"] = (
    bundle_rules["support"] *
    basket.shape[0]
)

bundle_rules[
    [
        "antecedent_name",
        "consequent_name",
        "support",
        "confidence",
        "lift",
        "estimated_target_orders"
    ]
].head(10)

,antecedent_name,consequent_name,support,confidence,lift,estimated_target_orders
80,Vantar Kitchen Towels x6,Aurelia Dish Soap 750ml,0.007326,0.789223,37.845144,996.0
81,Aurelia Dish Soap 750ml,Vantar Kitchen Towels x6,0.007326,0.351323,37.845144,996.0
114,Meridian Essentials Penne 500g,Zephyr Pesto 190g,0.004774,0.194195,36.363432,649.0
115,Zephyr Pesto 190g,Meridian Essentials Penne 500g,0.004774,0.893939,36.363432,649.0
67,Sundara Ketchup 500ml,Meridian Essentials Sliced Cheese x24,0.004038,0.399563,33.633830,549.0
66,Meridian Essentials Sliced Cheese x24,Sundara Ketchup 500ml,0.004038,0.339938,33.633830,549.0
64,Meridian Essentials Beef Burger Patties x4,Sundara Ketchup 500ml,0.009379,0.327427,32.395952,1275.0
65,Sundara Ketchup 500ml,Meridian Essentials Beef Burger Patties x4,0.009379,0.927948,32.395952,1275.0
60,Meridian Essentials Beef Burger Patties x4,Meridian Essentials Sliced Cheese x24,0.010872,0.379558,31.949878,1478.0
61,Meridian Essentials Sliced Cheese x24,Meridian Essentials Beef Burger Patties x4,0.010872,0.915170,31.949878,1478.0


In [74]:
bundle_rules["consequent_id"] = (
    bundle_rules["consequents"]
    .apply(lambda x: list(x)[0])
)

bundle_rules["consequent_price_aed"] = (
    bundle_rules["consequent_id"]
    .map(product_lookup["unit_price_aed"])
)

In [75]:
bundle_rules["estimated_revenue_opportunity_aed"] = (
    bundle_rules["antecedent_orders"] *
    bundle_rules["confidence"] *
    bundle_rules["consequent_price_aed"]
)

In [76]:
commercial_bundles = bundle_rules[
    (bundle_rules["support"] >= 0.005) &
    (bundle_rules["confidence"] >= 0.15) &
    (bundle_rules["lift"] >= 1.5)
].sort_values(
    "estimated_revenue_opportunity_aed",
    ascending=False
)

commercial_bundles[
    [
        "antecedent_name",
        "consequent_name",
        "support",
        "confidence",
        "lift",
        "estimated_target_orders",
        "consequent_price_aed",
        "estimated_revenue_opportunity_aed"
    ]
].head(10)

,antecedent_name,consequent_name,support,confidence,lift,estimated_target_orders,consequent_price_aed,estimated_revenue_opportunity_aed
80,Vantar Kitchen Towels x6,Aurelia Dish Soap 750ml,0.007326,0.789223,37.845144,996.0,11.45,NaN
81,Aurelia Dish Soap 750ml,Vantar Kitchen Towels x6,0.007326,0.351323,37.845144,996.0,14.45,NaN
64,Meridian Essentials Beef Burger Patties x4,Sundara Ketchup 500ml,0.009379,0.327427,32.395952,1275.0,22.45,NaN
65,Sundara Ketchup 500ml,Meridian Essentials Beef Burger Patties x4,0.009379,0.927948,32.395952,1275.0,21.45,NaN
60,Meridian Essentials Beef Burger Patties x4,Meridian Essentials Sliced Cheese x24,0.010872,0.379558,31.949878,1478.0,11.45,NaN
61,Meridian Essentials Sliced Cheese x24,Meridian Essentials Beef Burger Patties x4,0.010872,0.915170,31.949878,1478.0,21.45,NaN
14,Sundara Newborn Diapers x64,Peak & Pine Stage 1 Formula 900g,0.009401,0.586239,31.916779,1278.0,64.95,NaN
15,Peak & Pine Stage 1 Formula 900g,Sundara Newborn Diapers x64,0.009401,0.511814,31.916779,1278.0,111.00,NaN
69,Sundara Ketchup 500ml,Bright Harvest Brioche Buns x4,0.005333,0.527656,31.544529,725.0,7.45,NaN
68,Bright Harvest Brioche Buns x4,Sundara Ketchup 500ml,0.005333,0.318821,31.544529,725.0,22.45,NaN


In [77]:
top_3_bundles = commercial_bundles.head(3).copy()

top_3_bundles[
    [
        "antecedent_name",
        "consequent_name",
        "support",
        "confidence",
        "lift",
        "estimated_revenue_opportunity_aed"
    ]
]

,antecedent_name,consequent_name,support,confidence,lift,estimated_revenue_opportunity_aed
80,Vantar Kitchen Towels x6,Aurelia Dish Soap 750ml,0.007326,0.789223,37.845144,NaN
81,Aurelia Dish Soap 750ml,Vantar Kitchen Towels x6,0.007326,0.351323,37.845144,NaN
64,Meridian Essentials Beef Burger Patties x4,Sundara Ketchup 500ml,0.009379,0.327427,32.395952,NaN


In [78]:
segment_top_rules = []

for segment, rules_df in segment_rule_results.items():
    
    if rules_df.empty:
        continue
    
    top_rule = rules_df.iloc[0]
    
    segment_top_rules.append({
        "segment": segment,
        "antecedent": ", ".join(
            top_rule["antecedents_names"]
        ),
        "consequent": ", ".join(
            top_rule["consequents_names"]
        ),
        "support": top_rule["support"],
        "confidence": top_rule["confidence"],
        "lift": top_rule["lift"]
    })

segment_top_rules = pd.DataFrame(
    segment_top_rules
)

segment_top_rules

,segment,antecedent,consequent,support,confidence,lift
0,Family & Household Shoppers,Meridian Essentials Conditioner 400ml,Orchard Lane Shampoo 400ml,0.010453,0.921053,55.266459
1,Frequent Personal Care Shoppers,Vantar Kitchen Towels x6,Aurelia Dish Soap 750ml,0.004527,0.838488,67.456983
2,Health & Wellness Specialists,Meridian Essentials Hummus Dip 250g,Nordvik Tortilla Chips 200g,0.003218,0.643678,70.004023
3,Lapsed Everyday Shoppers,Orchard Lane Shampoo 400ml,Meridian Essentials Conditioner 400ml,0.007649,0.465517,58.688424
4,Premium Home & Living Shoppers,"Aurelia Hair Oil 200ml, Meridian Essentials Co...",Orchard Lane Shampoo 400ml,0.003719,0.993976,46.181890


## Segment-Aware Recommendations

The strongest product associations can differ across customer segments.

If segment-level rules show materially different product affinities, the
checkout recommendation should be segment-aware rather than showing the
same recommendation to every customer.

For example:

- Premium customers may respond to premium complementary products.
- Family-oriented customers may show stronger household and baby-product
  affinities.
- Health & Wellness customers may have stronger wellness-related
  combinations.

The final decision should be based on the actual segment-level rule output
rather than assumed behaviour.

In [79]:
top_rules[
    [
        "antecedents_names",
        "consequents_names",
        "support",
        "confidence",
        "lift",
        "leverage",
        "conviction"
    ]
].to_csv(
    TABLES_PATH / "sku_association_rules.csv",
    index=False
)

In [80]:
category_rules[
    [
        "antecedent",
        "consequent",
        "support",
        "confidence",
        "lift",
        "leverage",
        "conviction"
    ]
].to_csv(
    TABLES_PATH / "category_association_rules.csv",
    index=False
)

In [81]:
subcategory_rules[
    [
        "antecedent",
        "consequent",
        "support",
        "confidence",
        "lift",
        "leverage",
        "conviction"
    ]
].to_csv(
    TABLES_PATH / "subcategory_association_rules.csv",
    index=False
)

In [82]:
recommendation_comparison.to_csv(
    TABLES_PATH / "recommendation_model_comparison.csv",
    index=False
)

commercial_bundles[
    [
        "antecedent_name",
        "consequent_name",
        "support",
        "confidence",
        "lift",
        "estimated_target_orders",
        "consequent_price_aed",
        "estimated_revenue_opportunity_aed"
    ]
].to_csv(
    TABLES_PATH / "bundle_opportunities.csv",
    index=False
)

In [83]:
if not segment_top_rules.empty:
    segment_top_rules.to_csv(
        TABLES_PATH / "segment_association_rules.csv",
        index=False
    )

In [84]:
similarity_records = []

for product_id in item_similarity_df.index:
    
    similar = get_similar_products(
        product_id,
        top_n=10
    )
    
    for rank, (_, row) in enumerate(
        similar.iterrows(),
        start=1
    ):
        
        similarity_records.append({
            "product_id": product_id,
            "recommended_product_id": row["product_id"],
            "rank": rank,
            "similarity": row["similarity"]
        })

similarity_output = pd.DataFrame(
    similarity_records
)

similarity_output.to_csv(
    TABLES_PATH / "item_item_similarity.csv",
    index=False
)

print(
    "Similarity recommendations saved:",
    len(similarity_output)
)

Similarity recommendations saved: 3910


In [85]:
print("=" * 75)
print("NOTEBOOK 04 VALIDATION")
print("=" * 75)

print(
    f"Training orders:              {basket.shape[0]:,}"
)

print(
    f"Products in baskets:          {basket.shape[1]:,}"
)

print(
    f"Frequent itemsets:             {len(frequent_itemsets):,}"
)

print(
    f"Association rules:             {len(rules):,}"
)

print(
    f"Meaningful pair rules:         {len(pair_rules):,}"
)

print(
    f"Interesting cross-category rules: "
    f"{len(interesting_rules):,}"
)

print("-" * 75)

print(
    f"Recommendation evaluation customers: "
    f"{len(evaluation_customers):,}"
)

print("-" * 75)

print("Recommendation performance:")

display(
    recommendation_comparison.round(4)
)

print("-" * 75)

if not commercial_bundles.empty:
    
    print("Top 3 bundle opportunities:")
    
    display(
        top_3_bundles[
            [
                "antecedent_name",
                "consequent_name",
                "support",
                "confidence",
                "lift",
                "estimated_revenue_opportunity_aed"
            ]
        ].round(4)
    )

print("-" * 75)

print("Saved outputs:")
print("- sku_association_rules.csv")
print("- category_association_rules.csv")
print("- subcategory_association_rules.csv")
print("- segment_association_rules.csv")
print("- bundle_opportunities.csv")
print("- recommendation_model_comparison.csv")
print("- item_item_similarity.csv")

print("=" * 75)
print("Notebook 04 completed.")
print("=" * 75)

NOTEBOOK 04 VALIDATION
Training orders:              135,945
Products in baskets:          391
Frequent itemsets:             387
Association rules:             224
Meaningful pair rules:         116
Interesting cross-category rules: 70
---------------------------------------------------------------------------
Recommendation evaluation customers: 5,832
---------------------------------------------------------------------------
Recommendation performance:


,model,precision_at_5,recall_at_10,hit_rate_at_5
0,Popularity Baseline,0.1187,0.0976,0.4417
1,Item-Item Cosine,0.1166,0.0921,0.4331
2,Truncated SVD,0.1045,0.0846,0.4009


---------------------------------------------------------------------------
Top 3 bundle opportunities:


,antecedent_name,consequent_name,support,confidence,lift,estimated_revenue_opportunity_aed
80,Vantar Kitchen Towels x6,Aurelia Dish Soap 750ml,0.0073,0.7892,37.8451,NaN
81,Aurelia Dish Soap 750ml,Vantar Kitchen Towels x6,0.0073,0.3513,37.8451,NaN
64,Meridian Essentials Beef Burger Patties x4,Sundara Ketchup 500ml,0.0094,0.3274,32.3960,NaN


---------------------------------------------------------------------------
Saved outputs:
- sku_association_rules.csv
- category_association_rules.csv
- subcategory_association_rules.csv
- segment_association_rules.csv
- bundle_opportunities.csv
- recommendation_model_comparison.csv
- item_item_similarity.csv
Notebook 04 completed.


In [88]:
commercial_bundles.columns.tolist()

['antecedents',
 'consequents',
 'antecedent support',
 'consequent support',
 'support',
 'confidence',
 'lift',
 'representativity',
 'leverage',
 'conviction',
 'zhangs_metric',
 'jaccard',
 'certainty',
 'kulczynski',
 'antecedents_names',
 'consequents_names',
 'antecedent_id',
 'consequent_id',
 'product_id_x',
 'antecedent_name',
 'brand_x',
 'antecedent_category',
 'antecedent_subcategory',
 'unit_price_aed_x',
 'product_id_y',
 'consequent_name',
 'brand_y',
 'consequent_category',
 'consequent_subcategory',
 'unit_price_aed_y',
 'orders_with_rule',
 'antecedent_orders',
 'estimated_target_orders',
 'consequent_price_aed',
 'estimated_revenue_opportunity_aed']

In [90]:
commercial_bundles[
    [
        "antecedent_name",
        "consequent_name",
        "support",
        "confidence",
        "lift",
        "orders_with_rule",
        "antecedent_orders",
        "estimated_target_orders",
        "consequent_price_aed",
        "estimated_revenue_opportunity_aed"
    ]
].head(10)

,antecedent_name,consequent_name,support,confidence,lift,orders_with_rule,antecedent_orders,estimated_target_orders,consequent_price_aed,estimated_revenue_opportunity_aed
80,Vantar Kitchen Towels x6,Aurelia Dish Soap 750ml,0.007326,0.789223,37.845144,996.0,NaN,996.0,11.45,NaN
81,Aurelia Dish Soap 750ml,Vantar Kitchen Towels x6,0.007326,0.351323,37.845144,996.0,NaN,996.0,14.45,NaN
64,Meridian Essentials Beef Burger Patties x4,Sundara Ketchup 500ml,0.009379,0.327427,32.395952,1275.0,NaN,1275.0,22.45,NaN
65,Sundara Ketchup 500ml,Meridian Essentials Beef Burger Patties x4,0.009379,0.927948,32.395952,1275.0,NaN,1275.0,21.45,NaN
60,Meridian Essentials Beef Burger Patties x4,Meridian Essentials Sliced Cheese x24,0.010872,0.379558,31.949878,1478.0,NaN,1478.0,11.45,NaN
61,Meridian Essentials Sliced Cheese x24,Meridian Essentials Beef Burger Patties x4,0.010872,0.915170,31.949878,1478.0,NaN,1478.0,21.45,NaN
14,Sundara Newborn Diapers x64,Peak & Pine Stage 1 Formula 900g,0.009401,0.586239,31.916779,1278.0,NaN,1278.0,64.95,NaN
15,Peak & Pine Stage 1 Formula 900g,Sundara Newborn Diapers x64,0.009401,0.511814,31.916779,1278.0,NaN,1278.0,111.00,NaN
69,Sundara Ketchup 500ml,Bright Harvest Brioche Buns x4,0.005333,0.527656,31.544529,725.0,NaN,725.0,7.45,NaN
68,Bright Harvest Brioche Buns x4,Sundara Ketchup 500ml,0.005333,0.318821,31.544529,725.0,NaN,725.0,22.45,NaN


In [91]:
commercial_bundles[
    [
        "estimated_target_orders",
        "consequent_price_aed",
        "estimated_revenue_opportunity_aed"
    ]
].isna().sum()

estimated_target_orders               0
consequent_price_aed                  0
estimated_revenue_opportunity_aed    50
dtype: int64

In [92]:
commercial_bundles["estimated_revenue_opportunity_aed"] = (
    commercial_bundles["estimated_target_orders"]
    * commercial_bundles["consequent_price_aed"]
)

In [93]:
commercial_bundles[
    [
        "antecedent_name",
        "consequent_name",
        "support",
        "confidence",
        "lift",
        "estimated_target_orders",
        "consequent_price_aed",
        "estimated_revenue_opportunity_aed"
    ]
].head(10)

,antecedent_name,consequent_name,support,confidence,lift,estimated_target_orders,consequent_price_aed,estimated_revenue_opportunity_aed
80,Vantar Kitchen Towels x6,Aurelia Dish Soap 750ml,0.007326,0.789223,37.845144,996.0,11.45,11404.20
81,Aurelia Dish Soap 750ml,Vantar Kitchen Towels x6,0.007326,0.351323,37.845144,996.0,14.45,14392.20
64,Meridian Essentials Beef Burger Patties x4,Sundara Ketchup 500ml,0.009379,0.327427,32.395952,1275.0,22.45,28623.75
65,Sundara Ketchup 500ml,Meridian Essentials Beef Burger Patties x4,0.009379,0.927948,32.395952,1275.0,21.45,27348.75
60,Meridian Essentials Beef Burger Patties x4,Meridian Essentials Sliced Cheese x24,0.010872,0.379558,31.949878,1478.0,11.45,16923.10
61,Meridian Essentials Sliced Cheese x24,Meridian Essentials Beef Burger Patties x4,0.010872,0.915170,31.949878,1478.0,21.45,31703.10
14,Sundara Newborn Diapers x64,Peak & Pine Stage 1 Formula 900g,0.009401,0.586239,31.916779,1278.0,64.95,83006.10
15,Peak & Pine Stage 1 Formula 900g,Sundara Newborn Diapers x64,0.009401,0.511814,31.916779,1278.0,111.00,141858.00
69,Sundara Ketchup 500ml,Bright Harvest Brioche Buns x4,0.005333,0.527656,31.544529,725.0,7.45,5401.25
68,Bright Harvest Brioche Buns x4,Sundara Ketchup 500ml,0.005333,0.318821,31.544529,725.0,22.45,16276.25


In [94]:
top_3_bundles = (
    commercial_bundles[
        (commercial_bundles["support"] >= 0.005) &
        (commercial_bundles["confidence"] >= 0.20)
    ]
    .sort_values(
        ["lift", "support"],
        ascending=[False, False]
    )
    .head(3)
    .copy()
)

In [95]:
top_3_bundles[
    [
        "antecedent_name",
        "consequent_name",
        "support",
        "confidence",
        "lift",
        "estimated_revenue_opportunity_aed"
    ]
]

,antecedent_name,consequent_name,support,confidence,lift,estimated_revenue_opportunity_aed
80,Vantar Kitchen Towels x6,Aurelia Dish Soap 750ml,0.007326,0.789223,37.845144,11404.20
81,Aurelia Dish Soap 750ml,Vantar Kitchen Towels x6,0.007326,0.351323,37.845144,14392.20
64,Meridian Essentials Beef Burger Patties x4,Sundara Ketchup 500ml,0.009379,0.327427,32.395952,28623.75


In [96]:
# Create an unordered pair key so A→B and B→A are treated as one bundle
commercial_bundles["pair_key"] = commercial_bundles.apply(
    lambda row: tuple(sorted([
        row["antecedent_id"],
        row["consequent_id"]
    ])),
    axis=1
)

# Keep the strongest direction for each product pair
unique_bundles = (
    commercial_bundles
    .sort_values(
        ["pair_key", "confidence"],
        ascending=[True, False]
    )
    .drop_duplicates("pair_key")
)

# Select commercially meaningful bundles
top_3_bundles = (
    unique_bundles[
        (unique_bundles["support"] >= 0.005) &
        (unique_bundles["confidence"] >= 0.20)
    ]
    .sort_values(
        ["lift", "support"],
        ascending=[False, False]
    )
    .head(3)
    .copy()
)

top_3_bundles[
    [
        "antecedent_name",
        "consequent_name",
        "support",
        "confidence",
        "lift",
        "estimated_revenue_opportunity_aed"
    ]
]

,antecedent_name,consequent_name,support,confidence,lift,estimated_revenue_opportunity_aed
80,Vantar Kitchen Towels x6,Aurelia Dish Soap 750ml,0.007326,0.789223,37.845144,11404.20
65,Sundara Ketchup 500ml,Meridian Essentials Beef Burger Patties x4,0.009379,0.927948,32.395952,27348.75
61,Meridian Essentials Sliced Cheese x24,Meridian Essentials Beef Burger Patties x4,0.010872,0.915170,31.949878,31703.10


In [97]:
commercial_bundles.drop(columns=["pair_key"], errors="ignore").to_csv(
    TABLES_PATH / "bundle_opportunities.csv",
    index=False
)

print("Bundle opportunities saved successfully.")

Bundle opportunities saved successfully.


In [102]:
# ================================================================
# NOTEBOOK 04 — FINAL VALIDATION
# ================================================================

print("=" * 75)
print("NOTEBOOK 04 VALIDATION")
print("=" * 75)


# ------------------------------------------------
# 2. Recommendation Evaluation
# ------------------------------------------------

print(
    f"Recommendation evaluation customers: "
    f"{len(evaluation_customers):,}"
)

print("-" * 75)

print("Recommendation performance:")

print(
    recommendation_comparison[
        [
            "model",
            "precision_at_5",
            "recall_at_10",
            "hit_rate_at_5"
        ]
    ].to_string(index=False)
)

print("-" * 75)

# ------------------------------------------------
# 3. Top Bundle Opportunities
# ------------------------------------------------

# Treat A → B and B → A as the same product pair
commercial_bundles["pair_key"] = commercial_bundles.apply(
    lambda row: tuple(sorted([
        row["antecedent_id"],
        row["consequent_id"]
    ])),
    axis=1
)

# Keep the stronger direction based on confidence
unique_bundles = (
    commercial_bundles
    .sort_values(
        ["pair_key", "confidence"],
        ascending=[True, False]
    )
    .drop_duplicates("pair_key")
)

# Select commercially meaningful bundles
top_3_bundles = (
    unique_bundles[
        (unique_bundles["support"] >= 0.005) &
        (unique_bundles["confidence"] >= 0.20)
    ]
    .sort_values(
        ["lift", "support"],
        ascending=[False, False]
    )
    .head(3)
    .copy()
)

print("Top 3 bundle opportunities:")

print(
    top_3_bundles[
        [
            "antecedent_name",
            "consequent_name",
            "support",
            "confidence",
            "lift",
            "estimated_revenue_opportunity_aed"
        ]
    ].to_string(index=False)
)

print("-" * 75)

# ------------------------------------------------
# 4. Save Corrected Bundle Opportunities
# ------------------------------------------------

commercial_bundles.drop(
    columns=["pair_key"],
    errors="ignore"
).to_csv(
    TABLES_PATH / "bundle_opportunities.csv",
    index=False
)

# ------------------------------------------------
# 5. Saved Outputs
# ------------------------------------------------

print("Saved outputs:")
print("- sku_association_rules.csv")
print("- category_association_rules.csv")
print("- subcategory_association_rules.csv")
print("- segment_association_rules.csv")
print("- bundle_opportunities.csv")
print("- recommendation_model_comparison.csv")
print("- item_item_similarity.csv")

# ------------------------------------------------
# 6. Final Status
# ------------------------------------------------

print("=" * 75)
print("Notebook 04 completed.")
print("=" * 75)

NOTEBOOK 04 VALIDATION
Recommendation evaluation customers: 5,832
---------------------------------------------------------------------------
Recommendation performance:
              model  precision_at_5  recall_at_10  hit_rate_at_5
Popularity Baseline        0.118690      0.097602       0.441701
   Item-Item Cosine        0.116598      0.092058       0.433128
      Truncated SVD        0.104458      0.084558       0.400892
---------------------------------------------------------------------------
Top 3 bundle opportunities:
                      antecedent_name                            consequent_name  support  confidence      lift  estimated_revenue_opportunity_aed
             Vantar Kitchen Towels x6                    Aurelia Dish Soap 750ml 0.007326    0.789223 37.845144                           11404.20
                Sundara Ketchup 500ml Meridian Essentials Beef Burger Patties x4 0.009379    0.927948 32.395952                           27348.75
Meridian Essentials Slice